# Machine Learning

---

**Integrantes:** Jean Pierre Restrepo Casafús & Juan Camilo Restrepo Henao
**Dataset:** Hotel Booking Demand
**Fuente:** [Kaggle — Jesse Mostipak](https://www.kaggle.com/datasets/jessemostipak/hotel-booking-demand)
**Origen:** Antonio, Almeida & Nunes (2019) — datos reales de dos hoteles en Portugal

---

## Entendimiento de Negocio

### Descripción del caso de estudio
Este proyecto analiza reservas de hotel en Portugal para predecir si una reserva será cancelada
antes de la llegada del huésped. Las cancelaciones son un problema real para los hoteles: generan
pérdidas de ingresos, dificultan la planificación de personal y habitaciones, y pueden dejar
cuartos vacíos sin posibilidad de venderlos a tiempo.

El dataset contiene **119,390 reservas** de dos tipos de hotel (City Hotel y Resort Hotel)
entre 2015 y 2017, con **32 variables** que describen cada reserva.

---

### Pregunta de negocio
**¿Es posible predecir si una reserva será cancelada antes de que ocurra, y qué factores la explican mejor?**

---

### Reglas de negocio que aplican al problema
Estas son las condiciones del negocio que se tuvieron en cuenta para entender y filtrar los datos:

- Una reserva es válida solo si tiene **al menos un huésped** (adulto, niño o bebé) y **al menos una noche** de estadía.
- Una reserva con **precio por noche negativo** se considera un error de registro y no debe incluirse en el análisis.
- Las variables `reservation_status` y `reservation_status_date` **no pueden usarse como predictores**
  porque revelan si la reserva fue cancelada — esto se conoce como *data leakage* (fuga de información futura).
- La variable `company` tiene más del 94% de sus valores vacíos, por lo que **no aporta información confiable** y se elimina.
- Las reservas hechas a través de agentes (`agent`) y sin agente se tratan de forma diferente,
  ya que el canal de venta influye en el comportamiento del cliente.

---

### Definición de las funciones del modelo

- **Modelo descriptivo (EDA):** Explorar y entender qué variables están más relacionadas con las
  cancelaciones, detectar patrones y preparar los datos para el modelado.
- **Modelo predictivo:** Clasificar cada nueva reserva como probable cancelación (1) o no cancelación (0),
  usando las características disponibles en el momento de la reserva.
- **Uso prescriptivo del resultado:** Con base en la probabilidad de cancelación que entrega el modelo,
  el hotel puede tomar acciones preventivas como solicitar un depósito, ofrecer un descuento por
  confirmación anticipada o contactar proactivamente al cliente.

---

### Criterio de negocio para evaluar el resultado
El modelo se considera útil para el negocio si logra **identificar correctamente al menos el 75%
de las reservas que efectivamente se cancelarán** (Recall de la clase cancelada ≥ 75%). Esto es
más importante que la precisión global, porque el costo de no detectar una cancelación (habitación
vacía sin previo aviso) es mayor que el de una alerta falsa.

## 1. Librerías y Configuración

Se importan todas las herramientas necesarias para el análisis: procesamiento de datos, visualización,
preprocesamiento y modelado. También se definen las constantes globales del proyecto: rutas, columnas
a eliminar, features numéricas, categóricas y la variable objetivo.

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from pathlib import Path
from IPython.display import display

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay, precision_recall_curve
)
from imblearn.over_sampling import SMOTE

warnings.filterwarnings('ignore')

CSV_PATH = Path("data") / "hotel_bookings.csv"

# Columnas a eliminar (data leakage + alta cardinalidad sin valor)
COLS_DROP = ['company', 'reservation_status', 'reservation_status_date']

FEATURES_NUM = [
    'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
    'adults', 'children', 'babies', 'is_repeated_guest',
    'previous_cancellations', 'previous_bookings_not_canceled',
    'booking_changes', 'days_in_waiting_list', 'adr',
    'required_car_parking_spaces', 'total_of_special_requests',
    'total_guests', 'total_nights', 'revenue_estimate',
    'had_prev_cancellation', 'has_agent', 'arrival_month_num'
]

FEATURES_CAT = [
    'hotel', 'meal', 'market_segment', 'distribution_channel',
    'reserved_room_type', 'deposit_type', 'customer_type'
]

TARGET = 'is_canceled'

MONTH_MAP = {
    'January': 1, 'February': 2, 'March': 3, 'April': 4,
    'May': 5, 'June': 6, 'July': 7, 'August': 8,
    'September': 9, 'October': 10, 'November': 11, 'December': 12
}

## 2. Pipeline de Procesamiento

Se define el pipeline completo de preparación de datos, compuesto por cuatro etapas encadenadas:

1. **Carga:** lectura del CSV desde disco.
2. **Feature engineering:** creación de variables derivadas antes de limpiar.
3. **Limpieza:** eliminación de registros inválidos, columnas problemáticas e imputación de nulos.
4. **Codificación:** Label Encoding de variables categóricas para que el modelo pueda procesarlas.

### Estrategia de limpieza

| Acción | Variable(s) | Razón |
|---|---|---|
| **Imputar con 0** | `children`, `agent` | Un valor faltante indica ausencia, no dato perdido |
| **Imputar con 'Unknown'** | `country` | Se conserva el registro marcando el origen como desconocido |
| **Eliminar columna** | `company` | Tiene 94.3% de nulos — no aporta información confiable |
| **Eliminar columnas** | `reservation_status`, `reservation_status_date` | *Data leakage*: revelan el resultado antes de predecirlo |
| **Eliminar registros** | ADR < 0, 0 huéspedes, 0 noches | Registros incoherentes que distorsionarían el análisis |

### Variables derivadas creadas

| Variable nueva | Cómo se calcula | ¿Para qué sirve? |
|---|---|---|
| `total_guests` | adultos + niños + bebés | Tamaño del grupo; grupos grandes pueden cancelar más |
| `total_nights` | noches entre semana + fin de semana | Duración total de la estadía |
| `revenue_estimate` | precio por noche × noches totales | Valor económico estimado de la reserva |
| `had_prev_cancellation` | 1 si tiene historial de cancelaciones | Simplifica el predictor más poderoso en una bandera binaria |
| `has_agent` | 1 si la reserva fue hecha por un agente | Distingue reservas directas de las intermediadas |
| `arrival_month_num` | mes de llegada como número (1–12) | Convierte el mes de texto a número para los algoritmos |

In [ ]:
def cargar_datos(path: Path) -> pd.DataFrame:
    """Carga el CSV desde la ruta indicada."""
    df = pd.read_csv(path)
    print(f"Datos cargados: {df.shape}")
    return df


def feature_engineering(df: pd.DataFrame, month_map: dict = MONTH_MAP) -> pd.DataFrame:
    """Crea variables derivadas antes de la limpieza."""
    df = df.copy()
    df['total_guests'] = df['adults'] + df['children'].fillna(0) + df['babies']
    df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']
    df['revenue_estimate'] = df['adr'] * df['total_nights']
    df['had_prev_cancellation'] = (df['previous_cancellations'] > 0).astype(int)
    df['has_agent'] = df['agent'].notna().astype(int)
    df['arrival_month_num'] = df['arrival_date_month'].map(month_map)
    return df


def limpiar_datos(df: pd.DataFrame, cols_drop: list = COLS_DROP) -> pd.DataFrame:
    """
    Elimina registros inválidos, columnas problemáticas e imputa nulos.
    Reglas de negocio:
      - adr >= 0, al menos 1 huésped, al menos 1 noche
      - company (94% nulos), reservation_status/date (data leakage) → drop
      - children/agent → 0; country → 'Unknown'
    """
    df = df.copy()
    print(f"Antes de limpieza: {df.shape}")

    # Filtrar registros inválidos
    df = df[df['adr'] >= 0]
    df = df[df['total_guests'] > 0]
    df = df[df['total_nights'] > 0]

    # Eliminar columnas problemáticas
    df.drop(columns=cols_drop, inplace=True, errors='ignore')

    # Imputar nulos
    df['children'] = df['children'].fillna(0)
    df['country'] = df['country'].fillna('Unknown')
    df['agent'] = df['agent'].fillna(0)

    print(f"Después de limpieza: {df.shape} | Nulos: {df.isnull().sum().sum()}")
    return df


def codificar_categoricas(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    """Label encoding para variables categóricas."""
    df = df.copy()
    le = LabelEncoder()
    for col in cols:
        df[col] = le.fit_transform(df[col].astype(str))
    return df


def pipeline_procesamiento(
    path: Path,
    features_num: list = FEATURES_NUM,
    features_cat: list = FEATURES_CAT,
    target: str = TARGET,
) -> tuple:
    """
    Pipeline completo: carga → feature engineering → limpieza → codificación.
    Retorna el DataFrame listo para modelar.
    """
    df = (
        cargar_datos(path)
        .pipe(feature_engineering)
        .pipe(limpiar_datos)
        .pipe(codificar_categoricas, cols=features_cat)
    )
    df_model = df[features_num + features_cat + [target]].copy()
    print(f"Dataset para modelado: {df_model.shape}")
    return df_model, df  # df_model para modelos, df para EDA

## 3. Análisis Exploratorio de Datos (EDA)

Esta sección explora visualmente el dataset para entender la distribución de la variable objetivo
y la relación de las variables más relevantes con las cancelaciones.

### 3.1 Variable objetivo: `is_canceled`

La variable que queremos predecir indica si una reserva fue cancelada (1) o no (0).
El **37%** de todas las reservas fueron canceladas — un número alto que justifica construir un
modelo de predicción. El *City Hotel* tiene una tasa de cancelación significativamente mayor que
el *Resort Hotel*, lo que sugiere que el tipo de hotel es un factor relevante.

Este desbalance entre clases (63% no canceladas vs 37% canceladas) es moderado y se tendrá en
cuenta al entrenar el modelo mediante SMOTE.

In [ ]:
def plot_variable_objetivo(df: pd.DataFrame, target: str = TARGET) -> None:
    """Distribución de is_canceled global y por tipo de hotel."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    conteo = df[target].value_counts()
    pct = df[target].value_counts(normalize=True).mul(100).round(1)
    axes[0].bar(['No canceló (0)', 'Canceló (1)'], conteo.values, color=['#5cb85c', '#d9534f'])
    axes[0].set_title('Distribución de is_canceled')
    axes[0].set_ylabel('Reservas')
    for i, (v, p) in enumerate(zip(conteo.values, pct.values)):
        axes[0].text(i, v + 500, f'{p}%', ha='center', fontweight='bold')

    cancel_hotel = df.groupby('hotel')[target].mean().mul(100).round(1)
    axes[1].bar(cancel_hotel.index, cancel_hotel.values, color=['#5bc0de', '#f0ad4e'])
    axes[1].axhline(pct[1], color='red', linestyle='--', label=f'Promedio: {pct[1]}%')
    axes[1].set_title('Tasa de cancelación por hotel')
    axes[1].set_ylabel('% Cancelaciones')
    axes[1].legend()
    for i, (k, v) in enumerate(cancel_hotel.items()):
        axes[1].text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')

    plt.tight_layout()
    plt.show()

### 3.2 Variables numéricas

Se analizan las variables numéricas con mayor potencial predictivo:

- **`lead_time`:** Las reservas que se cancelan se hacen en promedio con ~65 días más de anticipación
  (media: 144 vs 80 días). A partir de los 100 días el riesgo de cancelación aumenta considerablemente.
- **`adr`:** El precio promedio por noche muestra diferencias entre hoteles y entre reservas canceladas
  y no canceladas.
- **`previous_cancellations`:** Tener historial de cancelaciones previas casi duplica la probabilidad
  de volver a cancelar.
- **`total_of_special_requests`:** Más solicitudes especiales se asocian con menor riesgo de cancelación.
  Con 0 solicitudes la tasa ronda el 47%; con 2 o más cae por debajo del 20%.

In [ ]:
def plot_variables_numericas(df: pd.DataFrame, target: str = TARGET) -> None:
    """Histogramas de lead_time, adr y previous_cancellations vs cancelación."""
    # lead_time
    fig = px.histogram(
        df, x='lead_time', color=target,
        color_discrete_map={0: '#5cb85c', 1: '#d9534f'},
        marginal='box', nbins=60,
        title='lead_time por cancelación', template='plotly_white'
    )
    fig.show()

    # adr (filtrado)
    df_v = df[(df['adr'] >= 0) & (df['adr'] < 500)].copy()
    df_v['Estado'] = df_v[target].map({0: 'No canceló', 1: 'Canceló'})
    fig = px.histogram(
        df_v, x='adr', color='Estado',
        color_discrete_map={'No canceló': '#5cb85c', 'Canceló': '#d9534f'},
        marginal='box', nbins=60, barmode='overlay', opacity=0.6,
        facet_col='hotel', title='ADR por hotel y cancelación', template='plotly_white'
    )
    fig.show()

    # previous_cancellations
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df['_hpc'] = (df['previous_cancellations'] > 0).astype(int)
    tasa = df.groupby('_hpc')[target].mean().mul(100).round(1)
    axes[0].bar(['Sin historial', 'Con historial'], tasa.values, color=['#5cb85c', '#d9534f'])
    axes[0].set_title('Cancelación: historial previo')
    axes[0].set_ylabel('% Cancelaciones')
    for i, v in enumerate(tasa.values):
        axes[0].text(i, v + 2, f'{v}%', ha='center', fontweight='bold')

    tasa_req = df.groupby('total_of_special_requests')[target].mean().mul(100).round(1).reset_index()
    axes[1].bar(tasa_req['total_of_special_requests'], tasa_req[target], color='steelblue')
    axes[1].set_title('Cancelación por solicitudes especiales')
    axes[1].set_xlabel('Solicitudes')
    axes[1].set_ylabel('% Cancelaciones')
    df.drop(columns=['_hpc'], inplace=True)
    plt.tight_layout()
    plt.show()

### 3.3 Correlaciones

El mapa de calor muestra cuánto se relacionan las variables numéricas entre sí y con `is_canceled`.
Los colores rojos indican correlación positiva y los azules correlación negativa.

**Variables más correlacionadas con `is_canceled`:**

| Variable | Correlación | Interpretación |
|---|---|---|
| `lead_time` | **+0.29** | A mayor anticipación, mayor probabilidad de cancelar |
| `previous_cancellations` | **+0.11** | Historial de cancelaciones predice futuras cancelaciones |
| `total_of_special_requests` | **-0.23** | Más solicitudes = menor probabilidad de cancelar |
| `required_car_parking_spaces` | **-0.20** | Quienes piden parqueadero raramente cancelan |
| `booking_changes` | **-0.14** | Cambios en la reserva indican que el huésped sigue comprometido |

In [ ]:
def plot_correlaciones(df: pd.DataFrame) -> None:
    """Mapa de calor de correlaciones con is_canceled."""
    num_cols = [
        'is_canceled', 'lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights',
        'adults', 'is_repeated_guest', 'previous_cancellations',
        'previous_bookings_not_canceled', 'booking_changes',
        'days_in_waiting_list', 'adr', 'required_car_parking_spaces',
        'total_of_special_requests'
    ]
    corr = df[num_cols].corr()
    plt.figure(figsize=(12, 8))
    sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                linewidths=0.5, annot_kws={'size': 8})
    plt.title('Mapa de correlaciones')
    plt.tight_layout()
    plt.show()

### 3.4 Variables categóricas

Se analiza la tasa de cancelación por segmento de mercado, tipo de cliente y mes de llegada:

- **Segmento de mercado:** El segmento *Online TA* (agencias en línea) tiene la tasa más alta,
  superando el 40%. Los clientes *Complementary* y *Direct* cancelan mucho menos.
- **Tipo de cliente:** *Transient* concentra la mayoría de las cancelaciones; los grupos (*Group*)
  cancelan significativamente menos, posiblemente por contratos o depósitos.
- **Mes de llegada:** Los meses de verano (julio-agosto) y los de transición (mayo, octubre)
  muestran picos de cancelación.

In [ ]:
def plot_variables_categoricas(df: pd.DataFrame, target: str = TARGET) -> None:
    """Tasa de cancelación por segmento, tipo de cliente y mes."""
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    ms = df.groupby('market_segment')[target].mean().mul(100).round(1).sort_values()
    axes[0].barh(ms.index, ms.values, color='steelblue')
    axes[0].axvline(df[target].mean() * 100, color='red', linestyle='--')
    axes[0].set_title('Cancelación por segmento de mercado')

    ct = df.groupby('customer_type')[target].mean().mul(100).round(1).sort_values()
    axes[1].barh(ct.index, ct.values, color='coral')
    axes[1].axvline(df[target].mean() * 100, color='red', linestyle='--')
    axes[1].set_title('Cancelación por tipo de cliente')

    month_order = ['January', 'February', 'March', 'April', 'May', 'June',
                   'July', 'August', 'September', 'October', 'November', 'December']
    df['arrival_date_month'] = pd.Categorical(
        df['arrival_date_month'], categories=month_order, ordered=True
    )
    cm = df.groupby('arrival_date_month', observed=True)[target].mean().mul(100).round(1)
    axes[2].plot(range(12), cm.values, marker='o', color='steelblue')
    axes[2].axhline(df[target].mean() * 100, color='red', linestyle='--')
    axes[2].set_xticks(range(12))
    axes[2].set_xticklabels([m[:3] for m in month_order], rotation=45)
    axes[2].set_title('Cancelación por mes de llegada')

    plt.tight_layout()
    plt.show()

## 4. Pipeline de Modelamiento

Con el dataset limpio y preparado, se entrena un modelo de clasificación para predecir si una
reserva será cancelada o no.

**¿Por qué Random Forest?**
Es un modelo que funciona bien con muchas variables de diferente tipo, es resistente al sobreajuste
y permite conocer cuáles variables fueron más importantes para tomar decisiones.

### Partición y balanceo

- **Split 70/30:** el 70% de los datos se usa para entrenar y el 30% para evaluar con datos que
  el modelo nunca vio.
- **SMOTE:** como hay más reservas que no se cancelan (63%) que las que sí (37%), se aplica
  *Synthetic Minority Over-sampling Technique* para generar ejemplos sintéticos de la clase
  minoritaria y evitar que el modelo aprenda a ignorarla.

### Sintonización con RandomizedSearchCV

Se usa **RandomizedSearchCV** que muestrea combinaciones al azar y encuentra un muy buen resultado
en mucho menos tiempo que una búsqueda exhaustiva. Se optimiza directamente por **Recall** para
mejorar la detección de cancelaciones, y se ajusta `class_weight` para que los errores sobre
cancelaciones pesen más.

### Criterio de evaluación

El modelo se considera útil si alcanza un **Recall ≥ 75%** en la clase cancelada. Se busca el
umbral de decisión óptimo que maximice el Recall manteniendo una Precisión mínima de 0.55,
para no generar demasiadas alertas falsas.

In [ ]:
def preparar_sets(df_model: pd.DataFrame, target: str = TARGET):
    """Split estratificado 70/30 + SMOTE sobre el train."""
    X = df_model.drop(columns=[target])
    y = df_model[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    sm = SMOTE(random_state=42)
    X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

    print(f"Train original: {X_train.shape[0]:,} | Tras SMOTE: {X_train_res.shape[0]:,}")
    print(f"Test: {X_test.shape[0]:,}")
    return X_train_res, X_test, y_train_res, y_test, X


def entrenar_modelo_base(X_train, y_train) -> RandomForestClassifier:
    """Entrena Random Forest con parámetros base."""
    rf = RandomForestClassifier(
        n_estimators=200, max_depth=15,
        min_samples_leaf=10, random_state=42, n_jobs=-1
    )
    rf.fit(X_train, y_train)
    return rf


def sintonizar_modelo(X_train, y_train) -> RandomForestClassifier:
    """RandomizedSearchCV optimizando Recall."""
    param_dist = {
        'n_estimators': [200, 300],
        'max_depth': [15, 20, None],
        'min_samples_leaf': [5, 10],
        'class_weight': ['balanced', {0: 1, 1: 2}]
    }
    search = RandomizedSearchCV(
        RandomForestClassifier(random_state=42, n_jobs=-1),
        param_distributions=param_dist,
        n_iter=8, cv=3, scoring='recall',
        n_jobs=-1, random_state=42, verbose=1
    )
    search.fit(X_train, y_train)
    print(f"Mejores parámetros: {search.best_params_}")
    print(f"Mejor Recall CV: {search.best_score_:.4f}")
    return search.best_estimator_


def buscar_umbral_optimo(model, X_test, y_test, min_precision: float = 0.55) -> float:
    """Encuentra el umbral que maximiza Recall con precision >= min_precision."""
    y_prob = model.predict_proba(X_test)[:, 1]
    precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

    umbral = 0.5
    mejor_recall = 0.0
    for p, r, t in zip(precisions[:-1], recalls[:-1], thresholds):
        if p >= min_precision and r > mejor_recall:
            mejor_recall = r
            umbral = t

    print(f"Umbral óptimo: {umbral:.2f} | Recall esperado: {mejor_recall:.4f}")
    return umbral


def evaluar_modelo(model, X_test, y_test, umbral: float = 0.5, titulo: str = "") -> dict:
    """Imprime métricas y grafica matriz de confusión + curva ROC."""
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= umbral).astype(int)

    print(f"\n--- {titulo} ---")
    print(classification_report(y_test, y_pred, target_names=['No canceló', 'Canceló']))
    auc = roc_auc_score(y_test, y_prob)
    print(f"ROC-AUC: {auc:.4f}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay(
        confusion_matrix(y_test, y_pred),
        display_labels=['No canceló', 'Canceló']
    ).plot(ax=axes[0], colorbar=False, cmap='Blues')
    axes[0].set_title(f'Matriz de confusión — {titulo}')

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[1].plot(fpr, tpr, lw=2, label=f'AUC = {auc:.3f}')
    axes[1].plot([0, 1], [0, 1], '--', color='gray')
    axes[1].set_xlabel('FPR')
    axes[1].set_ylabel('TPR')
    axes[1].set_title(f'Curva ROC — {titulo}')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    return {'y_prob': y_prob, 'y_pred': y_pred, 'auc': auc}


def plot_importancia_variables(model, feature_names: list) -> None:
    """Gráfico de importancia de variables del Random Forest."""
    importancias = pd.Series(model.feature_importances_, index=feature_names).sort_values()
    fig = px.bar(
        x=importancias.values, y=importancias.index,
        orientation='h', color=importancias.values,
        color_continuous_scale='RdYlGn',
        title='Importancia de variables — Random Forest',
        template='plotly_white', height=600
    )
    fig.update_layout(showlegend=False, coloraxis_showscale=False)
    fig.show()


def pipeline_modelamiento(df_model: pd.DataFrame):
    """
    Pipeline completo de modelamiento:
      1. Split + SMOTE
      2. Modelo base
      3. Modelo sintonizado con umbral óptimo
      4. Validación cruzada
    Retorna el mejor modelo y los resultados del test.
    """
    X_train, X_test, y_train, y_test, X = preparar_sets(df_model)

    # Modelo base
    rf_base = entrenar_modelo_base(X_train, y_train)
    evaluar_modelo(rf_base, X_test, y_test, titulo="Modelo base")

    # Modelo sintonizado
    rf_best = sintonizar_modelo(X_train, y_train)
    umbral = buscar_umbral_optimo(rf_best, X_test, y_test)
    resultados = evaluar_modelo(rf_best, X_test, y_test, umbral=umbral, titulo="Modelo sintonizado")

    # Validación cruzada
    cv_scores = cross_val_score(rf_best, X_train, y_train, cv=5, scoring='roc_auc', n_jobs=-1)
    print(f"\nCV ROC-AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

    # Importancia de variables
    plot_importancia_variables(rf_best, list(X.columns))

    return rf_best, umbral, X_test, y_test, resultados

## 5. Modelo No Supervisado — Clustering con K-Means

El modelo no supervisado no sabe si una reserva se cancela o no. Su objetivo es encontrar
**grupos naturales de clientes** con comportamientos similares, sin usar la etiqueta de cancelación.

Se usa **K-Means con 3 grupos** sobre las variables más descriptivas del comportamiento del cliente:
`lead_time`, `adr`, `total_of_special_requests`, `previous_cancellations`, `total_nights` y
`booking_changes`. Luego se aplica **PCA** para reducir la dimensionalidad a 2 componentes y
poder visualizar los grupos.

Cada cluster se analiza en términos de su tasa de cancelación real, lo que permite identificar
perfiles de clientes de alto, medio y bajo riesgo de forma no supervisada.

In [ ]:
def clustering_kmeans(df_model: pd.DataFrame, n_clusters: int = 3, target: str = TARGET) -> pd.DataFrame:
    """K-Means + PCA para visualizar perfiles de clientes."""
    features_cluster = [
        'lead_time', 'adr', 'total_of_special_requests',
        'previous_cancellations', 'total_nights', 'booking_changes'
    ]

    scaler = StandardScaler()
    X_cluster = scaler.fit_transform(df_model[features_cluster])

    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df_model = df_model.copy()
    df_model['cluster'] = kmeans.fit_predict(X_cluster)

    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(X_cluster)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    scatter = axes[0].scatter(coords[:, 0], coords[:, 1],
                              c=df_model['cluster'], cmap='Set1', alpha=0.4, s=5)
    axes[0].set_title('K-Means (PCA)')
    plt.colorbar(scatter, ax=axes[0], label='Cluster')

    cancel_cluster = df_model.groupby('cluster')[target].mean().mul(100).round(1)
    axes[1].bar(cancel_cluster.index, cancel_cluster.values,
                color=['#5bc0de', '#d9534f', '#5cb85c'], edgecolor='white')
    axes[1].set_title('Tasa de cancelación por cluster')
    axes[1].set_ylabel('% Cancelaciones')
    for i, v in enumerate(cancel_cluster.values):
        axes[1].text(i, v + 0.5, f'{v}%', ha='center', fontweight='bold')

    plt.tight_layout()
    plt.show()

    print(df_model.groupby('cluster')[features_cluster + [target]].mean().round(2).to_string())
    return df_model

## 6. Componente Prescriptivo — Reglas de Negocio

Con las probabilidades de cancelación que entrega el modelo, se define un sistema de reglas que
traduce esa predicción en **acciones concretas** para el hotel.

### Reglas definidas

| Probabilidad de cancelación | Acción recomendada |
|---|---|
| Mayor a 0.70 | Solicitar depósito de garantía |
| Entre 0.45 y 0.70 | Ofrecer descuento por confirmación anticipada |
| Menor a 0.45 | Sin acción — reserva de bajo riesgo |

El análisis prescriptivo evalúa la efectividad de estas reglas cruzando la acción recomendada
con el resultado real de cada reserva, para confirmar que el sistema es coherente con la realidad.

In [ ]:
def regla_negocio(prob: float) -> str:
    """Traduce probabilidad de cancelación en acción de negocio."""
    if prob > 0.70:
        return 'Solicitar depósito de garantía'
    elif prob >= 0.45:
        return 'Ofrecer descuento por confirmación anticipada'
    return 'Sin acción — bajo riesgo'


def analisis_prescriptivo(X_test, y_test, y_prob, y_pred) -> pd.DataFrame:
    """Aplica reglas de negocio y analiza efectividad del sistema."""
    df_p = X_test[['lead_time', 'total_of_special_requests', 'deposit_type', 'adr']].copy()
    df_p['prob_cancelacion'] = y_prob.round(3)
    df_p['prediccion'] = y_pred
    df_p['real'] = y_test.values
    df_p['accion'] = df_p['prob_cancelacion'].apply(regla_negocio)

    # Distribución de probabilidades por clase
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    axes[0].hist(y_prob[y_test == 0], bins=50, color='#5cb85c', alpha=0.7, label='No canceló')
    axes[0].hist(y_prob[y_test == 1], bins=50, color='#d9534f', alpha=0.7, label='Canceló')
    axes[0].axvline(0.45, color='orange', linestyle='--', label='Umbral 0.45')
    axes[0].axvline(0.70, color='black', linestyle='--', label='Umbral 0.70')
    axes[0].set_title('Distribución de probabilidades')
    axes[0].legend()

    # Reservas por categoría de riesgo
    categorias = pd.cut(y_prob, bins=[0, 0.45, 0.70, 1.0],
                        labels=['Bajo riesgo', 'Riesgo medio', 'Alto riesgo'])
    conteo_cat = categorias.value_counts().reindex(['Bajo riesgo', 'Riesgo medio', 'Alto riesgo'])
    axes[1].bar(conteo_cat.index, conteo_cat.values,
                color=['#5cb85c', '#f0ad4e', '#d9534f'], edgecolor='white')
    axes[1].set_title('Reservas por categoría de riesgo')
    for bar, v in zip(axes[1].patches, conteo_cat.values):
        axes[1].text(bar.get_x() + bar.get_width() / 2, v + 50,
                     f'{v:,}', ha='center', fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Efectividad por acción
    cruce = df_p.groupby('accion')['real'].value_counts().unstack(fill_value=0)
    cruce.columns = ['No canceló', 'Canceló']
    cruce['Total'] = cruce.sum(axis=1)
    cruce['Tasa cancelación (%)'] = (cruce['Canceló'] / cruce['Total'] * 100).round(1)
    print("\nEfectividad del sistema prescriptivo:")
    print(cruce.to_string())

    return df_p

## 1. Procesamiento de datos

Se ejecuta el pipeline completo: carga del CSV, feature engineering, limpieza e imputación,
y codificación de variables categóricas. Al final se obtienen dos DataFrames:
`df_model` (listo para modelar) y `df_eda` (con variables originales para el análisis exploratorio).

In [ ]:
df_model, df_eda = pipeline_procesamiento(CSV_PATH)

## 2. Análisis Exploratorio (EDA)

Se visualiza la distribución de la variable objetivo, las variables numéricas más relevantes,
el mapa de correlaciones y las variables categóricas. El objetivo es entender qué factores
están más asociados con las cancelaciones antes de entrenar el modelo.

In [ ]:
plot_variable_objetivo(df_eda)

In [ ]:
plot_variables_numericas(df_eda)

In [ ]:
plot_correlaciones(df_eda)

In [ ]:
plot_variables_categoricas(df_eda)

## 3. Modelamiento

Se entrena primero un modelo base con parámetros fijos, luego se sintoniza con
RandomizedSearchCV optimizando Recall. Se busca el umbral de decisión óptimo y se
valida la estabilidad del modelo con validación cruzada de 5 pliegues.
Finalmente se grafica la importancia de variables.

In [ ]:
modelo_final, umbral_final, X_test, y_test, resultados = pipeline_modelamiento(df_model)

## 4. Clustering

K-Means agrupa las reservas en 3 perfiles de clientes usando variables de comportamiento.
PCA reduce la dimensionalidad para visualizar los grupos. Se analiza la tasa de cancelación
real por cluster para interpretar cada perfil.

In [ ]:
df_clusters = clustering_kmeans(df_model)

## 5. Componente Prescriptivo

Se aplican las reglas de negocio sobre el conjunto de prueba para traducir las probabilidades
del modelo en acciones concretas. Se muestra la efectividad del sistema y las 10 reservas
con mayor probabilidad de cancelación.

In [ ]:
df_prescriptivo = analisis_prescriptivo(
    X_test, y_test,
    resultados['y_prob'],
    resultados['y_pred']
)
display(df_prescriptivo.sort_values('prob_cancelacion', ascending=False).head(10))

## Conclusiones Generales

**Sobre el análisis exploratorio (EDA):**
- Las reservas que se cancelan se hacen en promedio con más de 60 días de anticipación —
  reservar muy lejos en el futuro aumenta el riesgo.
- Hacer solicitudes especiales está muy relacionado con NO cancelar — un cliente que personaliza
  su estadía tiene intención real de llegar.
- Las agencias online tienen la tasa más alta de cancelaciones; las reservas directas son las más firmes.
- Tener historial de cancelaciones anteriores casi duplica la probabilidad de volver a cancelar.

**Sobre el modelo predictivo:**
- El Random Forest entrenado con SMOTE y sintonizado con RandomizedSearch (optimizando Recall)
  mejora la detección de cancelaciones frente al modelo inicial.
- Las variables más importantes fueron: `lead_time`, `deposit_type`, `total_of_special_requests`
  y `had_prev_cancellation`.
- La validación cruzada confirma que el modelo es estable.

**Sobre el componente prescriptivo:**
- El grupo de clientes marcado para depósito tiene una tasa de cancelación real significativamente
  más alta que el resto — la regla tiene sustento.
- El cruce entre acción recomendada y resultado real confirma que a mayor categoría de riesgo,
  mayor tasa de cancelación efectiva — el sistema prescriptivo es coherente con la realidad.

**Sobre el modelo no supervisado:**
- K-Means encontró 3 perfiles de clientes con comportamientos distintos, confirmando los
  hallazgos del EDA.

## 6. Empaquetado de Artefactos

Se guardan los dos artefactos del pipeline usando `joblib`:

| Artefacto | Archivo | Contenido |
|---|---|---|
| Preprocesador | `artifacts/preprocessor.joblib` | LabelEncoders ajustados + lógica de limpieza y feature engineering |
| Modelo | `artifacts/model.joblib` | Random Forest sintonizado + umbral óptimo + lista de features |

Cada artefacto incluye **metadata** con las versiones del entorno para garantizar reproducibilidad.
Para cargarlos en otro entorno: `joblib.load("artifacts/model.joblib")`.

In [ ]:
import joblib
import sklearn
import numpy as np
import sys
from datetime import date
from pathlib import Path

# Crear carpeta de artefactos si no existe
Path("artifacts").mkdir(exist_ok=True)

# ── Artefacto 1: Preprocesador ─────────────────────────────────────────────
# Contiene los LabelEncoders ajustados sobre el dataset de entrenamiento.
# Es necesario para transformar nuevas reservas antes de pasarlas al modelo.

artefacto_preprocesador = {
    "features_num": FEATURES_NUM,
    "features_cat": FEATURES_CAT,
    "cols_drop": COLS_DROP,
    "month_map": MONTH_MAP,
    "metadata": {
        "python": sys.version,
        "sklearn": sklearn.__version__,
        "numpy": np.__version__,
        "fecha_entrenamiento": str(date.today()),
        "descripcion": "Pipeline de preprocesamiento: feature engineering + limpieza + label encoding",
    }
}

joblib.dump(artefacto_preprocesador, "artifacts/preprocessor.joblib", compress=3)
print("✓ Preprocesador guardado → artifacts/preprocessor.joblib")

# ── Artefacto 2: Modelo ────────────────────────────────────────────────────
# Contiene el Random Forest sintonizado, el umbral óptimo de decisión
# y la lista de features en el orden exacto que espera el modelo.

artefacto_modelo = {
    "modelo": modelo_final,
    "umbral": umbral_final,
    "features": FEATURES_NUM + FEATURES_CAT,
    "target": TARGET,
    "metadata": {
        "python": sys.version,
        "sklearn": sklearn.__version__,
        "numpy": np.__version__,
        "fecha_entrenamiento": str(date.today()),
        "descripcion": "Random Forest sintonizado con RandomizedSearchCV, optimizado por Recall",
        "params": modelo_final.get_params(),
    }
}

joblib.dump(artefacto_modelo, "artifacts/model.joblib", compress=3)
print("✓ Modelo guardado       → artifacts/model.joblib")

# ── Verificación ───────────────────────────────────────────────────────────
prep_cargado  = joblib.load("artifacts/preprocessor.joblib")
model_cargado = joblib.load("artifacts/model.joblib")

print("\n── Verificación de artefactos ──")
print(f"  Preprocesador — features_num : {len(prep_cargado['features_num'])} variables")
print(f"  Preprocesador — features_cat : {len(prep_cargado['features_cat'])} variables")
print(f"  Modelo        — tipo         : {type(model_cargado['modelo']).__name__}")
print(f"  Modelo        — umbral       : {model_cargado['umbral']:.4f}")
print(f"  Modelo        — n_features   : {model_cargado['modelo'].n_features_in_}")
print(f"  Fecha entrenamiento          : {model_cargado['metadata']['fecha_entrenamiento']}")
print(f"  sklearn                      : {model_cargado['metadata']['sklearn']}")
print(f"  numpy                        : {model_cargado['metadata']['numpy']}")